**Laboratorio de Métodos Cuantitativos Aplicados a la Gestión**

---

# **Clase 19 - Anonimización de datos**

## ¿Qué vamos a hacer en esta clase?

Vamos a explorar la **responsabilidad ética en el uso de datos** y aprender a **anonimizar datos personales** usando la librería `anonymizedf`, aplicando técnicas de sustitución controlada de información sensible.

| Parte | Dataset | Técnicas aplicadas |
|---|---|---|
| **Ejemplo 1** | Titanic | `fake_names`, `fake_ids`, `fake_decimal_numbers` |
| **Ejemplo 2** | Clientes (Superstore) | Anonimización encadenada con `chaining=True` |


---

# Ética y responsabilidad en el uso de datos

La anonimización es **una herramienta**, no una solución completa. El manejo ético de datos abarca principios y responsabilidades que van mucho más allá de ocultar nombres.

---

## ¿Por qué importa la ética en el análisis de datos?

Cada vez que trabajamos con datos sobre personas — empleados, clientes, pacientes, ciudadanos — tomamos decisiones que pueden afectar sus vidas. Un análisis mal planteado, un modelo sesgado o una publicación descuidada pueden:

- Vulnerar la **privacidad** de individuos
- Reproducir o amplificar **sesgos** (discriminación por género, edad, origen)
- Generar **decisiones injustas** en contratación, crédito, salud o justicia
- Violar la **confianza** de las personas que cedieron sus datos

---

## Principios éticos fundamentales

| Principio | ¿Qué significa en la práctica? |
|---|---|
| **Consentimiento informado** | Los datos solo deben usarse con el conocimiento y acuerdo de la persona. No alcanza con que estén disponibles. |
| **Finalidad** | Los datos recopilados para un propósito (ej. facturación) no deben reutilizarse para otro (ej. perfilado político) sin nuevo consentimiento. |
| **Minimización** | Recopilar solo los datos estrictamente necesarios. Si no necesitás el DNI, no lo pidas. |
| **Exactitud** | Mantener los datos actualizados y corregir errores que puedan perjudicar a las personas. |
| **Limitación del almacenamiento** | No conservar datos más tiempo del necesario. |
| **Transparencia** | Las personas deben poder saber qué datos se tienen sobre ellas y cómo se usan. |
| **Responsabilidad** | Quien trata datos es responsable del cumplimiento de todos estos principios. |

---

## Más allá de la anonimización: otros riesgos éticos

**1. Re-identificación**

Anonimizar no garantiza el anonimato. Cruzando variables aparentemente inocuas (código postal + fecha de nacimiento + género) se puede re-identificar al 87% de la población (Sweeney, 2000). La anonimización siempre debe evaluarse frente al riesgo real de re-identificación.

**2. Sesgo algorítmico**

Si los datos históricos reflejan discriminación pasada (ej. menos mujeres en puestos directivos), un modelo entrenado sobre esos datos **aprenderá y perpetuará esa discriminación**. La responsabilidad no termina con datos limpios: hay que auditar los resultados.

**3. Uso secundario no autorizado**

Publicar un dataset "anonimizado" no habilita su uso para cualquier fin. La finalidad original del dato sigue siendo relevante éticamente, aunque la persona ya no sea identificable.

**4. Privacidad diferencial**

Técnica avanzada que agrega **ruido matemático controlado** a los datos para garantizar privacidad incluso ante ataques sofisticados. Es el estándar de Apple, Google y el Censo de EE.UU. La anonimización simple (como la que haremos hoy) es un primer paso, no la solución definitiva.

---




In [ ]:
#instalamos el módulo que utilizaremos (1 vez)
!pip install anonymizedf --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 8.1 MB/s eta 0:00:00


In [ ]:
# Cargamos las librerías necesarias
import pandas as pd                          # manejo de DataFrames y archivos CSV
from anonymizedf.anonymizedf import anonymize # clase principal para anonimizar columnas

---

## Ejemplo 1

Dataset: Titanic — anonimizamos nombres, IDs y tarifas.

In [ ]:
# Importamos el conjunto de datos
!wget https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv

--2026-05-06 22:18:55--  https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.110.133, 185.199.111.133, 185.199.109.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.110.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 60302 (59K) [text/plain]
Saving to: ‘titanic.csv’

titanic.csv         100%[===================>]  58.89K  --.-KB/s    in 0.04s   

2026-05-06 22:18:55 (1.51 MB/s) - ‘titanic.csv’ saved [60302/60302]



In [ ]:
#leemos el archivo importado y generamos el DataFrame
archivo_1=pd.read_csv('titanic.csv')
#Lo vemos
archivo_1


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S
...,...,...,...,...,...,...,...,...,...,...,...,...
886,887,0,2,"Montvila, Rev. Juozas",male,27.0,0,0,211536,13.0000,NaN,S
887,888,1,1,"Graham, Miss. Margaret Edith",female,19.0,0,0,112053,30.0000,B42,S
888,889,0,3,"Johnston, Miss. Catherine Helen ""Carrie""",female,NaN,1,2,W./C. 6607,23.4500,NaN,S
889,890,1,1,"Behr, Mr. Karl Howell",male,26.0,0,0,111369,30.0000,C148,C


In [ ]:
# Preparamos el conjunto de datos para ser anonimizado
# Aquí se genera un objeto "an" de la clase anonymize,
#Este paso prepara el DataFrame para aplicar las funciones de anonimización de la librería anonymizedf.
an = anonymize(archivo_1)

# Seleccionamos las columnas que deseamos anonimizar
an.fake_names("Name")
an.fake_ids("PassengerId")
#se generarán nombres falsos y se asignará a los nombres originales.
#si hay nombres idénticos en la columna original,se les asignará el mismo nombre falso
#para mantener la consistencia en el conjunto de datos anonimizado.
archivo_1

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked,Fake_Name,Fake_PassengerId
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S,Clifford Hewitt-Martin,MKNV87261644605646
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C,Elaine Turnbull,AMFV91128830943448
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S,Brian Stephenson-Wilkinson,STJX03709914984620
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S,David Reed,NEGA30977894286768
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S,Eleanor Taylor,FCIG76383085596775
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
886,887,0,2,"Montvila, Rev. Juozas",male,27.0,0,0,211536,13.0000,NaN,S,Billy Carpenter,FWSG73799594673534
887,888,1,1,"Graham, Miss. Margaret Edith",female,19.0,0,0,112053,30.0000,B42,S,Judith Stewart,JEBF91714458483703
888,889,0,3,"Johnston, Miss. Catherine Helen ""Carrie""",female,NaN,1,2,W./C. 6607,23.4500,NaN,S,Dr Ross Burke,ANOT86150197292607
889,890,1,1,"Behr, Mr. Karl Howell",male,26.0,0,0,111369,30.0000,C148,C,Mr Martin Lynch,USGU51635236072810


In [ ]:
an.fake_decimal_numbers("Fare")
#Anonimiza columnas que contienen números decimales (como salarios, precios, mediciones).
#Sustituye los números decimales originales por números decimales falsos,
#manteniendo la coherencia para los valores repetidos.
archivo_1

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked,Fake_Name,Fake_PassengerId,Fake_Fare
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S,Clifford Hewitt-Martin,MKNV87261644605646,384.25
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C,Elaine Turnbull,AMFV91128830943448,378.98
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S,Brian Stephenson-Wilkinson,STJX03709914984620,31.54
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S,David Reed,NEGA30977894286768,459.22
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S,Eleanor Taylor,FCIG76383085596775,425.58
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
886,887,0,2,"Montvila, Rev. Juozas",male,27.0,0,0,211536,13.0000,NaN,S,Billy Carpenter,FWSG73799594673534,481.54
887,888,1,1,"Graham, Miss. Margaret Edith",female,19.0,0,0,112053,30.0000,B42,S,Judith Stewart,JEBF91714458483703,324.47
888,889,0,3,"Johnston, Miss. Catherine Helen ""Carrie""",female,NaN,1,2,W./C. 6607,23.4500,NaN,S,Dr Ross Burke,ANOT86150197292607,463.98
889,890,1,1,"Behr, Mr. Karl Howell",male,26.0,0,0,111369,30.0000,C148,C,Mr Martin Lynch,USGU51635236072810,324.47


### Otras funciones disponibles

---

```python
an.fake_whole_numbers()

an.fake_categories()

an.fake_dates()

an.fake_decimal_numbers()
```

---

### Consistencia para valores repetidos

Probemos que ocurre cuando en dos registros tenemos el mismo dato

In [ ]:
archivo_1.loc[0,"Name"]="Pepe Rodriguez"
archivo_1.loc[1,"Name"]="Pepe Rodriguez"

an = anonymize(archivo_1)
an.fake_names("Name")

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked,Fake_Name,Fake_PassengerId,Fake_Fare
0,1,0,3,Pepe Rodriguez,male,22.0,1,0,A/5 21171,7.2500,NaN,S,Mandy Watkins-Hamilton,MKNV87261644605646,384.25
1,2,1,1,Pepe Rodriguez,female,38.0,1,0,PC 17599,71.2833,C85,C,Mandy Watkins-Hamilton,AMFV91128830943448,378.98
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S,Toby Storey,STJX03709914984620,31.54
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S,Donna Robinson-Bradshaw,NEGA30977894286768,459.22
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S,Dr Ashleigh Cox,FCIG76383085596775,425.58
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
886,887,0,2,"Montvila, Rev. Juozas",male,27.0,0,0,211536,13.0000,NaN,S,Antony Taylor,FWSG73799594673534,481.54
887,888,1,1,"Graham, Miss. Margaret Edith",female,19.0,0,0,112053,30.0000,B42,S,Ashleigh Osborne-Roberts,JEBF91714458483703,324.47
888,889,0,3,"Johnston, Miss. Catherine Helen ""Carrie""",female,NaN,1,2,W./C. 6607,23.4500,NaN,S,Damien Pearson-Russell,ANOT86150197292607,463.98
889,890,1,1,"Behr, Mr. Karl Howell",male,26.0,0,0,111369,30.0000,C148,C,Mr Guy Hussain,USGU51635236072810,324.47


> **Nota:** Deberíamos conservar este conjunto de datos que nos permite identificar los datos anonimizados y generar un nuevo conjunto de datos que solo conserve las columnas anonimizadas además de aquellas de interés en caso de ser publicado/utilizado.

---

## Ejemplo 2

Dataset: Clientes (Superstore) — anonimización completa encadenada.

In [ ]:
# Cargamos el dataset de clientes desde la URL pública
# Fuente: data.world — dataset de clientes de una cadena de retail (Superstore)
archivo_2 = pd.read_csv("https://query.data.world/s/shcktxndtu3ojonm46tb5udlz7sp3e")
archivo_2

,Customer ID,Customer Name,Loyalty Reward Points,Segment,Date,Fraction
0,AA-10315,Alex Avila,76,Consumer,01/01/2000,7.6
1,AA-10375,Allen Armold,369,Consumer,02/01/2000,36.9
2,AA-10480,Andrew Allen,162,Consumer,03/01/2000,16.2
3,AA-10645,Anna Andreadi,803,Consumer,04/01/2000,80.3
4,AB-10015,Aaron Bergman,935,Consumer,05/01/2000,93.5
...,...,...,...,...,...,...
788,XP-21865,Xylona Preis,454,Consumer,27/02/2002,45.4
789,YC-21895,Yoseph Carroll,935,Corporate,28/02/2002,93.5
790,YS-21880,Yana Sorensen,829,Corporate,01/03/2002,82.9
791,ZC-21910,Zuschuss Carroll,745,Consumer,02/03/2002,74.5


In [ ]:
# Anonimización encadenada: se aplican múltiples transformaciones en una sola expresión
# chaining=True permite encadenar métodos con el operador punto (.)
# show_data_frame() devuelve el DataFrame modificado al final de la cadena
an_2 = anonymize(archivo_2)
archivo_2 = (
    an_2.fake_names("Customer Name", chaining=True)              # nombres ficticios
    .fake_ids("Customer ID", chaining=True)                      # IDs ficticios
    .fake_whole_numbers("Loyalty Reward Points", chaining=True)  # números enteros ficticios
    .fake_categories("Segment", chaining=True)                   # categorías ficticias
    .fake_dates("Date", chaining=True)                           # fechas ficticias
    .fake_decimal_numbers("Fraction", chaining=True)             # decimales ficticios
    .show_data_frame()                                           # devuelve el DataFrame
)
archivo_2

,Customer ID,Customer Name,Loyalty Reward Points,Segment,Date,Fraction,Fake_Customer Name,Fake_Customer ID,Fake_Loyalty Reward Points,Fake_Segment,Fake_Date,Fake_Fraction
0,AA-10315,Alex Avila,76,Consumer,01/01/2000,7.6,Shirley Begum,CDCC55127008599915,636,Segment 1,1999-02-18,49.02
1,AA-10375,Allen Armold,369,Consumer,02/01/2000,36.9,Miss Sally Bentley,NRWP27360819893463,357,Segment 1,1994-09-08,56.93
2,AA-10480,Andrew Allen,162,Consumer,03/01/2000,16.2,Ellie Bailey,PEDW44982433115229,447,Segment 1,2000-08-31,85.19
3,AA-10645,Anna Andreadi,803,Consumer,04/01/2000,80.3,Eleanor Kelly,JJZR85132260545130,723,Segment 1,2004-09-28,2.55
4,AB-10015,Aaron Bergman,935,Consumer,05/01/2000,93.5,Catherine Whittaker,RYOF69723886161177,486,Segment 1,2001-02-21,1.96
...,...,...,...,...,...,...,...,...,...,...,...,...
788,XP-21865,Xylona Preis,454,Consumer,27/02/2002,45.4,Vanessa Baker-Lee,GPGJ08059903106513,206,Segment 1,1995-03-05,46.66
789,YC-21895,Yoseph Carroll,935,Corporate,28/02/2002,93.5,Dr Terry Gibson,ZLIF04512985038309,486,Segment 3,2009-06-29,1.96
790,YS-21880,Yana Sorensen,829,Corporate,01/03/2002,82.9,James Bird,ZBSK50654867513937,667,Segment 3,1974-06-22,58.26
791,ZC-21910,Zuschuss Carroll,745,Consumer,02/03/2002,74.5,Philip Wilson,YQJS78807367021192,637,Segment 1,1976-01-27,48.92


## Responsabilidades concretas del analista

| Momento | Acción responsable |
|---|---|
| Antes de recibir datos | Verificar que existe consentimiento y base legal para el análisis |
| Al explorar | No publicar outputs que contengan datos personales sin anonimizar |
| Al modelar | Evaluar si el modelo puede discriminar o perjudicar grupos vulnerables |
| Al publicar | Usar solo las columnas mínimas necesarias; aplicar anonimización adecuada |
| Al archivar | Definir una política de retención: ¿hasta cuándo se guardan estos datos? |

---